# Raw player-season data

Downloads player-season tables from Baseball Savant, FanGraphs and the MLB Stats API into `data/raw/` and validates them. The Savant custom-leaderboard and percentile pulls build on Jeffrey's earlier scraping cells.

Reruns re-validate and reuse finished files. With every file present and `OVERWRITE = False`, a rerun makes no network requests. If a saved file fails validation, the run stops: inspect the file, then delete it or set `OVERWRITE = True`.

**Leakage:** every row is a full-season total. Don't use a season's row as pre-game information for games in that same season.

## Imports

In [1]:
import json
import re
import time
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

# Request sources directly, not through pybaseball, so responses can be checked before saving.

## Configuration

Each source defines its minimum differently: plate appearances on some boards, innings on others. The comments give each meaning, checked with small live requests.

All minimums are set to 0. Source coverage still varies. Apply qualification filters during modeling.

In [2]:
SEASONS = [2022, 2023, 2024, 2025, 2026]
# Find the repository root so data/raw/ is the same whether Jupyter starts there or in notebooks/.
REPO_ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                  if (folder / "requirements.txt").is_file()
                  and (folder / "notebooks" / "savant_scrape_updated.ipynb").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f"Could not find the repository root from {Path.cwd().name}/: no folder above it contains "
                            "requirements.txt and notebooks/savant_scrape_updated.ipynb. Start Jupyter inside the repository.")
RAW_DIR = REPO_ROOT / "data" / "raw"   # tracked in Git; *.partial files are ignored
OVERWRITE = False                # False: re-validate and reuse finished files

# Savant custom, batters: min = minimum PA. 0 and 1 return the same players.
SAVANT_CUSTOM_MIN_BATTER = 0
# Savant custom, pitchers: min = minimum innings, not PA. min=1 drops pitchers with 0.1-0.2 IP
# (7 in a 2026 check).
SAVANT_CUSTOM_MIN_PITCHER = 0
# Savant expected stats: min = minimum PA, for batters and pitchers.
SAVANT_XSTATS_MIN_PA = 0
# FanGraphs: qual = minimum PA (batting) or IP (pitching). At 0 the batting board also lists pitchers with 0 PA.
FANGRAPHS_QUAL = 0
# Savant percentile rankings take no minimum; Savant ranks only qualified players.

REQUEST_PAUSE_SECONDS = 2        # pause after each request; requests run one at a time
HEADERS = {"User-Agent": "mlb-game-simulator-research-notebook (python-requests)"}

RAW_DIR.mkdir(parents=True, exist_ok=True)
print("raw data folder:", RAW_DIR.relative_to(Path.cwd(), walk_up=True))

raw data folder: ..\data\raw


## Request and file helpers

- Every request has a timeout. Connection errors, timeouts, 429 and 5xx get up to three attempts and honor `Retry-After`. Other error statuses stop the run, so an access block is reported, not worked around.
- Savant can answer a bad request with an HTML page and status 200, so HTML and header-only CSVs are rejected.
- Each season table must have rows, valid player IDs, one row per player and only the requested season.
- Files are written to `.partial`, read back, then renamed. An interrupted run leaves only a `.partial` file.
- Saved files are re-validated before reuse. A file that fails stops the run instead of being replaced.

In [3]:
session = requests.Session()
session.headers.update(HEADERS)
RETRY_STATUSES = {429, 500, 502, 503, 504}
requests_made = []      # stays empty on a rerun with every file saved
RAW_TABLES = []         # for the validation section


def fetch(url, params, source, max_attempts=3):
    """GET with a timeout and status check, retrying transient failures."""
    for attempt in range(1, max_attempts + 1):
        requests_made.append(source)
        try:
            response = session.get(url, params=params, timeout=(15, 180))
        except (requests.ConnectionError, requests.Timeout) as error:
            problem, wait = f"{type(error).__name__}: {error}", 10 * attempt
        else:
            if response.status_code < 400:
                time.sleep(REQUEST_PAUSE_SECONDS)
                return response
            problem = f"HTTP {response.status_code}"
            if response.status_code not in RETRY_STATUSES:
                raise RuntimeError(f"{source}: {problem} from {response.url[:200]}")
            retry_after = response.headers.get("Retry-After", "")
            wait = min(int(retry_after), 300) if retry_after.isdigit() else 10 * attempt
        if attempt == max_attempts:
            raise RuntimeError(f"{source}: failed after {max_attempts} attempts ({problem})")
        print(f"  {source}: {problem}; retrying in {wait} s")
        time.sleep(wait)


def read_csv_response(response, source):
    """Parse a Savant CSV, rejecting HTML pages and header-only files."""
    body = response.content
    content_type = response.headers.get("Content-Type", "")
    if "html" in content_type or body.lstrip()[:1] == b"<":
        raise ValueError(f"{source}: expected CSV but received HTML ({content_type}); starts {body[:80]!r}")
    table = pd.read_csv(StringIO(body.decode("utf-8-sig")))   # Savant CSVs begin with a byte-order mark
    if table.empty:
        raise ValueError(f"{source}: the CSV contained no player rows")
    return table


def check_table(table, source, season, id_column, season_column, required_columns=()):
    """Raise ValueError listing every structural problem in a season table."""
    problems = []
    if table.empty:
        problems.append("no rows")
    missing = [c for c in [id_column, season_column, *required_columns] if c not in table.columns]
    if missing:
        problems.append(f"missing columns {missing}")
    else:
        ids = pd.to_numeric(table[id_column], errors="coerce")
        if ids.isna().any():
            problems.append(f"{ids.isna().sum()} missing or non-numeric {id_column}")
        elif (ids % 1 != 0).any():
            problems.append(f"non-integer {id_column}")
        if table[id_column].duplicated().any():
            problems.append(f"{table[id_column].duplicated().sum()} duplicate {id_column} (expected one row per player-season)")
        seasons = sorted(pd.unique(table[season_column]).tolist())
        if seasons != [season]:
            problems.append(f"{season_column} values {seasons[:5]}, expected [{season}]")
    if problems:
        raise ValueError(f"{source} {season}: " + "; ".join(problems))


def save_csv(table, path):
    """Write to <name>.partial, check it reads back identically, then rename it."""
    partial = path.with_name(path.name + ".partial")
    table.to_csv(partial, index=False)
    assert_same_values(pd.read_csv(partial), table)
    partial.replace(path)


def assert_same_values(left, right):
    """Assert two tables match cell by cell. None and NaN count as equal; a CSV can't tell them apart."""
    as_objects = lambda t: t.reset_index(drop=True).astype(object).where(t.reset_index(drop=True).notna(), None)
    pd.testing.assert_frame_equal(as_objects(left), as_objects(right), check_dtype=False, check_exact=False)


def load_or_download(path, download, validate):
    """Re-validate and reuse a finished file, or download, validate and save it."""
    if path.exists() and not OVERWRITE:
        table = pd.read_csv(path)
        try:
            validate(table)
        except ValueError as error:
            raise ValueError(f"{path} exists but failed validation, so it was neither reused nor replaced: "
                             f"{error}. Inspect it, then delete it or set OVERWRITE = True.") from error
        print(f"  reused {path.name}: {table.shape[0]} rows x {table.shape[1]} columns")
        return table
    table = download()
    validate(table)
    save_csv(table, path)
    print(f"  saved  {path.name}: {table.shape[0]} rows x {table.shape[1]} columns")
    return pd.read_csv(path)    # continue with exactly what is on disk


def register(source, role, season, path, table, id_column):
    RAW_TABLES.append({"source": source, "role": role, "season": season, "file": path.name,
                       "rows": table.shape[0], "columns": table.shape[1], "id_column": id_column, "table": table})

# Data acquisition

Each season table has one row per player per season. Traded players appear once, with combined totals. The sources define and include players differently, so each table is saved separately and nothing is merged here.

| Source | Files (per season) | Player ID column | Season column |
|---|---|---|---|
| Baseball Savant custom leaderboard | `savant_custom_{batter,pitcher}_{season}.csv` | `player_id` (MLBAM) | `year` |
| Baseball Savant expected statistics | `savant_expected_stats_{batter,pitcher}_{season}.csv` | `player_id` (MLBAM) | `year` |
| Baseball Savant percentile rankings (supplementary) | `savant_percentile_ranks_{batter,pitcher}_{season}.csv` | `player_id` (MLBAM) | `year` |
| FanGraphs major-league leaderboards | `fangraphs_{batting,pitching}_{season}.csv` | `playerid` (FanGraphs); `xMLBAMID` (MLBAM) | `Season` |
| MLB Stats API people lookup | `mlb_player_handedness.csv` (one file) | `player_id` (MLBAM) | not seasonal |

Join on MLBAM player ID and season: `player_id` and `year` in Savant, `xMLBAMID` and `Season` in FanGraphs. Each pair should be unique within a role and season, and validation checks this. Don't join on names: Savant writes "Last, First", FanGraphs writes "First Last", and names aren't unique.

## Savant custom leaderboard: batters

`https://baseballsavant.mlb.com/leaderboard/custom` with `csv=true` returns actual statistics. With `min=0` it lists every batter with at least one plate appearance. The CSV always adds `last_name, first_name`, `player_id` and `year`.

The field list is every batter field on the live page, grouped for readability. The 2022 and 2026 schemas matched when checked. `xera` and `r_stolen_base_pct` are left out because they were empty on the batter board in every season checked.

Units: `*_percent` and `*_rate` fields are percentages (24.4 means 24.4 %), speeds are mph, angles are degrees, and `avg_swing_length` is feet.

`pa` is Savant's Statcast-tracked plate-appearance count and `b_total_pa` is its standard count. They can differ by one or two, so validation compares FanGraphs' `PA` with both.

Bat tracking starts in 2023, so those fields are empty in 2022. Validation reports empty fields by season rather than filling them.

In [4]:
BATTER_FIELD_GROUPS = {
    "profile": ["player_age"],
    # Counts are kept with the rates so later work can re-aggregate.
    "results": ["pa", "ab", "hit", "single", "double", "triple", "home_run", "strikeout", "walk",
                "batting_avg", "slg_percent", "on_base_percent", "on_base_plus_slg", "isolated_power",
                "babip", "woba", "k_percent", "bb_percent"],
    # Savant's standard batting counts
    "batting_counts": ["b_ab_scoring", "b_ball", "b_called_strike", "b_catcher_interf", "b_foul",
                       "b_foul_tip", "b_game", "b_gnd_into_dp", "b_gnd_into_tp", "b_gnd_rule_double",
                       "b_hit_by_pitch", "b_hit_fly", "b_hit_ground", "b_hit_into_play", "b_hit_line_drive",
                       "b_hit_popup", "b_intent_ball", "b_intent_walk", "b_interference", "b_lob",
                       "b_out_fly", "b_out_ground", "b_out_line_drive", "b_out_popup", "b_pinch_hit",
                       "b_pinch_run", "b_pitchout", "b_played_dh", "b_rbi", "b_reached_on_error",
                       "b_reached_on_int", "b_sac_bunt", "b_sac_fly", "b_swinging_strike", "b_total_ball",
                       "b_total_bases", "b_total_pa", "b_total_pitches", "b_total_sacrifices",
                       "b_total_strike", "b_total_swinging_strike", "b_walkoff"],
    # x* = expected, from exit velocity and launch angle; *diff = actual minus expected
    "expected": ["xba", "xslg", "xwoba", "xobp", "xiso", "xwobacon", "xbacon", "wobacon", "bacon",
                 "xbadiff", "xslgdiff", "wobadiff"],
    "contact_quality": ["batted_ball", "exit_velocity_avg", "launch_angle_avg", "sweet_spot_percent",
                        "barrel", "barrel_batted_rate", "hard_hit_percent", "avg_best_speed",
                        "avg_hyper_speed", "solidcontact_percent", "flareburner_percent",
                        "poorlyunder_percent", "poorlytopped_percent", "poorlyweak_percent"],
    "batted_ball_profile": ["groundballs", "groundballs_percent", "flyballs", "flyballs_percent",
                            "linedrives", "linedrives_percent", "popups", "popups_percent",
                            "pull_percent", "straightaway_percent", "opposite_percent"],
    # z_ = in the strike zone, oz_ = outside it
    "plate_discipline": ["pitch_count", "swing_percent", "whiff_percent", "z_swing_percent",
                         "z_swing_miss_percent", "oz_swing_percent", "oz_swing_miss_percent",
                         "oz_contact_percent", "iz_contact_percent", "in_zone", "in_zone_percent",
                         "in_zone_swing", "in_zone_swing_miss", "out_zone", "out_zone_percent",
                         "out_zone_swing", "out_zone_swing_miss", "f_strike_percent", "edge",
                         "edge_percent", "meatball_percent", "meatball_swing_percent", "swords"],
    # from 2023
    "bat_tracking": ["avg_swing_speed", "avg_swing_length", "squared_up_contact", "squared_up_swing",
                     "blasts_contact", "blasts_swing", "fast_swing_rate", "attack_angle",
                     "attack_direction", "ideal_angle_rate", "vertical_swing_path"],
    # run values are in runs
    "pitch_group_run_value": ["pitch_count_fastball", "pitch_count_breaking", "pitch_count_offspeed",
                              "pitch_run_value_fastball", "pitch_run_value_breaking",
                              "pitch_run_value_offspeed", "swing_take_run_value"],
    "baserunning": ["sprint_speed", "hp_to_1b", "n_bolts", "runner_run_value", "r_run",
                    "r_caught_stealing_2b", "r_caught_stealing_3b", "r_caught_stealing_home",
                    "r_defensive_indiff", "r_interference", "r_pickoff_1b", "r_pickoff_2b",
                    "r_pickoff_3b", "r_stolen_base_2b", "r_stolen_base_3b", "r_stolen_base_home",
                    "r_total_caught_stealing", "r_total_pickoff", "r_total_stolen_base"],
    # Position-specific defense (catcher pop times, outfielder jumps), mostly empty for other players.
    # Kept in the raw data only.
    "defense_ancillary": ["n_outs_above_average", "fielding_run_value", "n_1star_percent",
                          "n_2star_percent", "n_3star_percent", "n_4star_percent", "n_5star_percent",
                          "n_fieldout_1stars", "n_fieldout_2stars", "n_fieldout_3stars",
                          "n_fieldout_4stars", "n_fieldout_5stars", "n_opp_1stars", "n_opp_2stars",
                          "n_opp_3stars", "n_opp_4stars", "n_opp_5stars", "f_bootup_distance",
                          "rel_league_bootup_distance", "rel_league_burst_distance",
                          "rel_league_reaction_distance", "rel_league_routing_distance",
                          "pop_2b_cs", "pop_2b_sb", "pop_2b_sba", "pop_2b_sba_count", "pop_3b_cs",
                          "pop_3b_sb", "pop_3b_sba", "pop_3b_sba_count", "exchange_2b_3b_sba",
                          "maxeff_arm_2b_3b_sba"],
}
BATTER_SELECTIONS = [field for group in BATTER_FIELD_GROUPS.values() for field in group]
print(len(BATTER_SELECTIONS), "batter fields in", len(BATTER_FIELD_GROUPS), "groups")

190 batter fields in 11 groups


## Savant custom leaderboard: pitchers

With `type=pitcher`, the same leaderboard returns each pitcher's own results and what batters did against them.

Here `min=0` means 0 innings. Pitchers who recorded at least one out are listed; pitchers who faced batters without recording an out are not. Position players who pitched are missing from this board but appear in expected statistics. Validation measures the gap.

`pa` is the Statcast-tracked batters-faced count and `p_total_pa` is the standard count. In the 2022-2026 check, FanGraphs' `TBF` matched `p_total_pa` for every matched pitcher, while `pa` differed by 1-2 for about 2 % of pitchers. Validation recomputes both comparisons on each run.

`p_formatted_ip` uses baseball notation: `61.1` is 61⅓ innings (184 outs) and `61.2` is 61⅔. It is stored as given and converted to outs only for calculations.

Per-pitch columns such as `{pitch}_avg_speed` and `n_{pitch}_formatted` are season summaries. They are empty, not zero, for pitches a pitcher doesn't throw. `fastball`, `breaking` and `offspeed` are group averages. `arm_angle` is in degrees.

Pitch codes: `ff` four-seam, `si` sinker, `fc` cutter, `sl` slider, `st` sweeper, `sv` slurve, `cu` curveball, `ch` changeup, `fs` splitter, `kn` knuckleball, `fo` forkball, `sc` screwball.

`fielding_run_value`, `runner_run_value`, `p_opp_batting_avg` and `p_opp_on_base_avg` are left out because they were empty on the pitcher board in every season checked. Opponent average and OBP are in `batting_avg` and `on_base_percent`.

In [5]:
# Pitch types with their own columns, then the three group averages
PITCH_CODES = ["ff", "si", "fc", "sl", "st", "sv", "cu", "ch", "fs", "kn", "fo", "sc",
               "fastball", "breaking", "offspeed"]

PITCHER_FIELD_GROUPS = {
    # arm_angle in degrees
    "profile": ["player_age", "pitch_hand", "arm_angle"],
    # p_formatted_ip is baseball innings notation
    "workload_and_record": ["p_game", "p_starting_p", "p_game_in_relief", "p_game_finished",
                            "p_complete_game", "p_shutout", "p_quality_start", "p_formatted_ip",
                            "p_out", "p_total_pa", "p_total_pitches", "p_win", "p_loss", "p_save",
                            "p_blown_save", "p_hold", "p_era", "p_earned_run", "p_unearned_run",
                            "p_run", "p_run_support", "p_inh_runner", "p_inh_runner_scored",
                            "p_beq_runner", "p_beq_runner_scored", "p_relief_no_out"],
    "outcomes_allowed": ["p_ab", "p_ab_scoring", "p_total_hits", "p_single", "p_double", "p_triple",
                         "p_home_run", "p_strikeout", "p_walk", "p_intent_walk", "p_hit_by_pitch",
                         "p_sac_bunt", "p_sac_fly", "p_total_sacrifices", "p_total_bases", "p_rbi",
                         "p_hit_scoring", "p_gnd_into_dp", "p_gnd_into_tp", "p_gnd_rule_double",
                         "p_reached_on_error", "p_catcher_interf", "p_lob", "p_walkoff"],
    "pitch_result_counts": ["p_ball", "p_called_strike", "p_foul", "p_foul_tip", "p_swinging_strike",
                            "p_total_ball", "p_total_strike", "p_total_swinging_strike",
                            "p_intent_ball", "p_automatic_ball", "p_pitchout", "p_missed_bunt",
                            "p_hit_fly", "p_hit_ground", "p_hit_into_play", "p_hit_line_drive",
                            "p_out_fly", "p_out_ground", "p_out_line_drive"],
    # while this pitcher was on the mound
    "running_game": ["p_balk", "p_wild_pitch", "p_passed_ball", "p_stolen_base_2b",
                     "p_stolen_base_3b", "p_stolen_base_home", "p_total_stolen_base",
                     "p_caught_stealing_2b", "p_caught_stealing_3b", "p_caught_stealing_home",
                     "p_total_caught_stealing", "p_defensive_indiff", "p_pickoff_1b", "p_pickoff_2b",
                     "p_pickoff_3b", "p_total_pickoff", "p_pickoff_attempt_1b", "p_pickoff_attempt_2b",
                     "p_pickoff_attempt_3b", "p_total_pickoff_attempt", "p_pickoff_error_1b",
                     "p_pickoff_error_2b", "p_pickoff_error_3b", "p_total_pickoff_error"],
    # Batter-board groups, measured against the pitcher
    "results_against": ["pa", "ab", "hit", "single", "double", "triple", "home_run", "strikeout",
                        "walk", "batting_avg", "slg_percent", "on_base_percent", "on_base_plus_slg",
                        "isolated_power", "babip", "woba", "k_percent", "bb_percent"],
    "expected_against": ["xba", "xslg", "xwoba", "xobp", "xiso", "xwobacon", "xbacon", "wobacon",
                         "bacon", "xbadiff", "xslgdiff", "wobadiff", "xera"],
    "contact_quality_against": ["batted_ball", "exit_velocity_avg", "launch_angle_avg",
                                "sweet_spot_percent", "barrel", "barrel_batted_rate",
                                "hard_hit_percent", "avg_best_speed", "avg_hyper_speed",
                                "solidcontact_percent", "flareburner_percent", "poorlyunder_percent",
                                "poorlytopped_percent", "poorlyweak_percent"],
    "batted_ball_profile_against": ["groundballs", "groundballs_percent", "flyballs",
                                    "flyballs_percent", "linedrives", "linedrives_percent", "popups",
                                    "popups_percent", "pull_percent", "straightaway_percent",
                                    "opposite_percent"],
    "plate_discipline_induced": ["pitch_count", "swing_percent", "whiff_percent", "z_swing_percent",
                                 "z_swing_miss_percent", "oz_swing_percent", "oz_swing_miss_percent",
                                 "oz_contact_percent", "iz_contact_percent", "in_zone",
                                 "in_zone_percent", "in_zone_swing", "in_zone_swing_miss", "out_zone",
                                 "out_zone_percent", "out_zone_swing", "out_zone_swing_miss",
                                 "f_strike_percent", "edge", "edge_percent", "meatball_percent",
                                 "meatball_swing_percent", "swords"],
    "bat_tracking_against": ["avg_swing_speed", "avg_swing_length", "squared_up_contact",
                             "squared_up_swing", "blasts_contact", "blasts_swing", "fast_swing_rate",
                             "attack_angle", "attack_direction", "ideal_angle_rate",
                             "vertical_swing_path"],
    "pitch_group_run_value": ["pitch_count_fastball", "pitch_count_breaking", "pitch_count_offspeed",
                              "pitch_run_value_fastball", "pitch_run_value_breaking",
                              "pitch_run_value_offspeed", "swing_take_run_value"],
    # n = pitch total behind these columns. n_{pitch}_formatted = usage %, avg_speed and range_speed = mph,
    # avg_spin = rpm, avg_break* = inches, z_induced = induced vertical break.
    "repertoire": ["n"] + [f"n_{code}_formatted" for code in PITCH_CODES] + [
        f"{code}_{stat}" for code in PITCH_CODES
        for stat in ["avg_speed", "avg_spin", "avg_break_x", "avg_break_z", "avg_break_z_induced",
                     "avg_break", "range_speed"]],
}
PITCHER_SELECTIONS = [field for group in PITCHER_FIELD_GROUPS.values() for field in group]
print(len(PITCHER_SELECTIONS), "pitcher fields in", len(PITCHER_FIELD_GROUPS), "groups")

314 pitcher fields in 13 groups


In [6]:
SAVANT_CUSTOM_URL = "https://baseballsavant.mlb.com/leaderboard/custom"
SAVANT_CUSTOM_SELECTIONS = {"batter": BATTER_SELECTIONS, "pitcher": PITCHER_SELECTIONS}
SAVANT_CUSTOM_MIN = {"batter": SAVANT_CUSTOM_MIN_BATTER, "pitcher": SAVANT_CUSTOM_MIN_PITCHER}
# Columns the CSV always adds, and schema fields not requested
SAVANT_CUSTOM_ID_COLUMNS = ["last_name, first_name", "player_id", "year"]
SAVANT_CUSTOM_NOT_REQUESTED = {
    "batter": {"player_id", "player_name", "year", "yearhidden", "xera", "r_stolen_base_pct"},
    "pitcher": {"player_id", "player_name", "year", "yearhidden", "fielding_run_value",
                "runner_run_value", "p_opp_batting_avg", "p_opp_on_base_avg"},
}


def savant_custom_schema(role, season):
    """Return Savant's custom-leaderboard field names, read from the records embedded in its page.

    The CSV export can't verify names: it returns any requested name, even a misspelled one, as an
    empty column."""
    page = fetch(SAVANT_CUSTOM_URL, {"year": season, "type": role}, f"Savant custom {role} {season} (schema)").text
    decoder = json.JSONDecoder()
    fields, records = set(), 0
    for match in re.finditer(r'"player_id"\s*:', page):
        try:
            record, _ = decoder.raw_decode(page, page.rfind("{", 0, match.start()))
        except ValueError:
            continue
        if isinstance(record, dict):
            fields |= set(record)
            records += 1
        if records == 10:
            break
    if records == 0:
        raise ValueError(f"Savant custom {role} {season}: no player records in the page, so field names could not be verified")
    return fields


def download_savant_custom(role, season):
    source = f"Savant custom {role} {season}"
    selections = SAVANT_CUSTOM_SELECTIONS[role]
    schema = savant_custom_schema(role, season)
    unknown = [field for field in selections if field not in schema]
    if unknown:
        raise ValueError(f"{source}: requested fields not in Savant's schema: {unknown}")
    unrequested = sorted(schema - set(selections) - SAVANT_CUSTOM_NOT_REQUESTED[role])
    if unrequested:
        print(f"  note: {source} schema has fields this notebook does not request yet: {unrequested}")
    params = {"year": season, "type": role, "min": SAVANT_CUSTOM_MIN[role],
              "selections": ",".join(selections), "chart": "false", "csv": "true"}
    table = read_csv_response(fetch(SAVANT_CUSTOM_URL, params, source), source)
    # A stray comma in the selections becomes an "Unnamed: n" column
    unexpected = [c for c in table.columns if c not in selections and c not in SAVANT_CUSTOM_ID_COLUMNS]
    if unexpected:
        raise ValueError(f"{source}: unexpected columns returned: {unexpected}")
    return table


def check_savant_custom(role, season):
    return lambda table: check_table(table, f"Savant custom {role}", season, "player_id", "year",
                                     SAVANT_CUSTOM_SELECTIONS[role])


savant_custom_batter, savant_custom_pitcher = {}, {}
for season in SEASONS:
    for role, tables in [("batter", savant_custom_batter), ("pitcher", savant_custom_pitcher)]:
        path = RAW_DIR / f"savant_custom_{role}_{season}.csv"
        tables[season] = load_or_download(path, lambda: download_savant_custom(role, season),
                                          check_savant_custom(role, season))
        register("Savant custom", role, season, path, tables[season], "player_id")

  reused savant_custom_batter_2022.csv: 693 rows x 193 columns
  reused savant_custom_pitcher_2022.csv: 813 rows x 317 columns
  reused savant_custom_batter_2023.csv: 656 rows x 193 columns
  reused savant_custom_pitcher_2023.csv: 805 rows x 317 columns
  reused savant_custom_batter_2024.csv: 651 rows x 193 columns
  reused savant_custom_pitcher_2024.csv: 806 rows x 317 columns
  reused savant_custom_batter_2025.csv: 673 rows x 193 columns
  reused savant_custom_pitcher_2025.csv: 812 rows x 317 columns
  reused savant_custom_batter_2026.csv: 662 rows x 193 columns
  reused savant_custom_pitcher_2026.csv: 819 rows x 317 columns


## Savant expected statistics

`/leaderboard/expected_statistics` returns actual and expected BA, SLG and wOBA (`est_*`, with `*_minus_*` differences), plus ERA and xERA for pitchers. Here `min` counts plate appearances.

Most values repeat custom-leaderboard fields. The table is kept because it covers different players: position players who pitched, and others the custom pitcher board omits. Validation compares the two player lists.

In [7]:
SAVANT_XSTATS_URL = "https://baseballsavant.mlb.com/leaderboard/expected_statistics"
XSTATS_REQUIRED = {
    "batter": ["last_name, first_name", "pa", "bip", "ba", "est_ba", "slg", "est_slg", "woba", "est_woba"],
    "pitcher": ["last_name, first_name", "pa", "bip", "ba", "est_ba", "slg", "est_slg", "woba", "est_woba",
                "era", "xera"],
}


def download_savant_xstats(role, season):
    source = f"Savant expected stats {role} {season}"
    params = {"type": role, "year": season, "position": "", "team": "", "min": SAVANT_XSTATS_MIN_PA, "csv": "true"}
    return read_csv_response(fetch(SAVANT_XSTATS_URL, params, source), source)


savant_xstats_batter, savant_xstats_pitcher = {}, {}
for season in SEASONS:
    for role, tables in [("batter", savant_xstats_batter), ("pitcher", savant_xstats_pitcher)]:
        path = RAW_DIR / f"savant_expected_stats_{role}_{season}.csv"
        validate = lambda table: check_table(table, f"Savant expected stats {role}", season,
                                             "player_id", "year", XSTATS_REQUIRED[role])
        tables[season] = load_or_download(path, lambda: download_savant_xstats(role, season), validate)
        register("Savant expected stats", role, season, path, tables[season], "player_id")

  reused savant_expected_stats_batter_2022.csv: 693 rows x 14 columns
  reused savant_expected_stats_pitcher_2022.csv: 871 rows x 17 columns
  reused savant_expected_stats_batter_2023.csv: 656 rows x 14 columns
  reused savant_expected_stats_pitcher_2023.csv: 863 rows x 17 columns
  reused savant_expected_stats_batter_2024.csv: 651 rows x 14 columns
  reused savant_expected_stats_pitcher_2024.csv: 855 rows x 17 columns
  reused savant_expected_stats_batter_2025.csv: 673 rows x 14 columns
  reused savant_expected_stats_pitcher_2025.csv: 873 rows x 17 columns
  reused savant_expected_stats_batter_2026.csv: 662 rows x 14 columns
  reused savant_expected_stats_pitcher_2026.csv: 868 rows x 17 columns


## Savant percentile rankings (supplementary)

`/leaderboard/percentile-rankings` returns rankings, not statistics. Each metric is a percentile from 1 to 100 among that season's qualified players: `xwoba = 90` means better than about 90 % of qualified batters, not an xwOBA of .900. The actual values are in the custom-leaderboard tables. The files are named `savant_percentile_ranks_*` to keep the two apart.

Savant ranks only qualified players, so many rows rank just a few metrics and many players are missing. These gaps are left as they are. Savant's export can include a placeholder row with no player name; such rows are dropped and reported.

In [8]:
SAVANT_PERCENTILE_URL = "https://baseballsavant.mlb.com/leaderboard/percentile-rankings"


def download_savant_percentiles(role, season):
    source = f"Savant percentile ranks {role} {season}"
    params = {"type": role, "year": season, "position": "", "team": "", "csv": "true"}
    table = read_csv_response(fetch(SAVANT_PERCENTILE_URL, params, source), source)
    placeholder = table["player_name"].isna()
    if placeholder.any():
        print(f"  {source}: dropped {placeholder.sum()} placeholder row(s) with no player name, "
              f"player_id {table.loc[placeholder, 'player_id'].tolist()}")
    return table.loc[~placeholder].reset_index(drop=True)


def check_percentiles(role, season):
    def validate(table):
        check_table(table, f"Savant percentile ranks {role}", season, "player_id", "year", ["player_name"])
        ranks = table.drop(columns=["player_name", "player_id", "year"]).apply(pd.to_numeric, errors="coerce")
        outside = int(((ranks < 0) | (ranks > 100)).sum().sum())
        if outside:
            raise ValueError(f"Savant percentile ranks {role} {season}: {outside} values outside 0-100")
    return validate


savant_percentiles_batter, savant_percentiles_pitcher = {}, {}
for season in SEASONS:
    for role, tables in [("batter", savant_percentiles_batter), ("pitcher", savant_percentiles_pitcher)]:
        path = RAW_DIR / f"savant_percentile_ranks_{role}_{season}.csv"
        tables[season] = load_or_download(path, lambda: download_savant_percentiles(role, season),
                                          check_percentiles(role, season))
        register("Savant percentile ranks", role, season, path, tables[season], "player_id")

  reused savant_percentile_ranks_batter_2022.csv: 628 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2022.csv: 704 rows x 22 columns
  reused savant_percentile_ranks_batter_2023.csv: 614 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2023.csv: 707 rows x 22 columns
  reused savant_percentile_ranks_batter_2024.csv: 606 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2024.csv: 696 rows x 22 columns
  reused savant_percentile_ranks_batter_2025.csv: 617 rows x 23 columns
  reused savant_percentile_ranks_pitcher_2025.csv: 712 rows x 22 columns
  reused savant_percentile_ranks_batter_2026.csv: 612 rows x 23 columns


  reused savant_percentile_ranks_pitcher_2026.csv: 706 rows x 22 columns


## FanGraphs leaderboards

When this notebook was written, `pb.batting_stats()` returned an empty table with only an `IDfg` column and no error. This section requests the public leaderboard page at `/leaders/major-league` instead, with its documented URL parameters (`stats`, `season`, `qual`, `pageitems`, `pagenum`, ...).

FanGraphs embeds the leaderboard as JSON (`__NEXT_DATA__`). This code depends on that undocumented format. If it changes, the cell raises an error instead of saving an empty table. FanGraphs' robots.txt does not disallow this path. Its terms of service ask users to access the site through its provided interface; this pull loads that public page a few times per season, one request at a time.

Every column is kept under its FanGraphs name. When checked, that was 475 batting and 544 pitching fields for every `type` value tested; validation shows the current counts. They include standard and advanced stats, WAR, wRC+, FIP/xFIP/SIERA, plate discipline, batted balls, FanGraphs pitch tracking (`pfx*`, `pi*`), pitch values, Stuff+/Location+/Pitching+ (`sp_*`) and `Bats`/`Throws`. `Name` and `Team` contain HTML links; `PlayerName` and `TeamNameAbb` are plain text. `IP` uses baseball notation (186.2 is 186⅔), and `xMLBAMID` is the MLBAM ID that links rows to Savant.

`team=0` gives one row per player, with traded players combined as "2 Tms" or "3 Tms"; the `playerid` uniqueness check confirms this. `qual=0` sets no minimum, so the batting board also lists pitchers with 0 PA. `month=0` is the full season. FanGraphs leaderboards default to the regular season, and validation compares PA with Savant. Each response's query key must match the requested filters, and pages are fetched until the row count equals `totalCount`.

In [9]:
FANGRAPHS_URL = "https://www.fangraphs.com/leaders/major-league"
FANGRAPHS_PAGE_SIZE = 2000      # normally one page per season
FANGRAPHS_MAX_PAGES = 10
FANGRAPHS_REQUIRED = {"bat": ["xMLBAMID", "PlayerName", "PA", "Bats"],
                      "pit": ["xMLBAMID", "PlayerName", "IP", "TBF", "Throws"]}


def fangraphs_page(stats, season, page_number):
    source = f"FanGraphs {stats} {season} page {page_number}"
    params = {"pos": "all", "stats": stats, "lg": "all", "qual": FANGRAPHS_QUAL, "type": 8,
              "season": season, "season1": season, "month": 0, "ind": 0, "team": 0,
              "pageitems": FANGRAPHS_PAGE_SIZE, "pagenum": page_number}
    html = fetch(FANGRAPHS_URL, params, source).text
    match = re.search(r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.S)
    if not match:
        raise ValueError(f"{source}: no embedded __NEXT_DATA__ block; the page layout may have changed")
    queries = json.loads(match.group(1))["props"]["pageProps"]["dehydratedState"]["queries"]
    boards = [q for q in queries if q.get("queryKey", [None])[0] == "leaders/major-league/data"]
    if len(boards) != 1:
        raise ValueError(f"{source}: expected one leaderboard dataset in the page, found {len(boards)}")
    applied, data = boards[0]["queryKey"][1], boards[0]["state"]["data"]
    # The query key echoes the filters FanGraphs applied; reject silent changes
    checked = ["stats", "season", "season1", "qual", "month", "ind", "team", "pageitems", "pagenum"]
    mismatched = {k: (applied.get(k), params[k]) for k in checked if str(applied.get(k)) != str(params[k])}
    if mismatched:
        raise ValueError(f"{source}: FanGraphs applied different filters (applied, requested): {mismatched}")
    return data["data"], data["totalCount"]


def download_fangraphs(stats, season):
    rows, total = fangraphs_page(stats, season, 1)
    page_number = 1
    while len(rows) < total:
        page_number += 1
        if page_number > FANGRAPHS_MAX_PAGES:
            raise ValueError(f"FanGraphs {stats} {season}: still incomplete after {FANGRAPHS_MAX_PAGES} pages")
        more, _ = fangraphs_page(stats, season, page_number)
        if not more:
            raise ValueError(f"FanGraphs {stats} {season}: page {page_number} was empty with {len(rows)} of {total} rows")
        rows += more
    if len(rows) != total:
        raise ValueError(f"FanGraphs {stats} {season}: collected {len(rows)} rows but totalCount is {total}")
    return pd.DataFrame(rows)


def check_fangraphs(stats, season):
    return lambda table: check_table(table, f"FanGraphs {stats}", season, "playerid", "Season",
                                     FANGRAPHS_REQUIRED[stats])


fangraphs_batting, fangraphs_pitching = {}, {}
for season in SEASONS:
    for stats, name, tables in [("bat", "batting", fangraphs_batting), ("pit", "pitching", fangraphs_pitching)]:
        path = RAW_DIR / f"fangraphs_{name}_{season}.csv"
        tables[season] = load_or_download(path, lambda: download_fangraphs(stats, season),
                                          check_fangraphs(stats, season))
        register("FanGraphs", "batter" if stats == "bat" else "pitcher", season, path, tables[season], "playerid")

  reused fangraphs_batting_2022.csv: 1495 rows x 475 columns


  reused fangraphs_pitching_2022.csv: 871 rows x 544 columns
  reused fangraphs_batting_2023.csv: 1457 rows x 475 columns
  reused fangraphs_pitching_2023.csv: 863 rows x 544 columns


  reused fangraphs_batting_2024.csv: 1454 rows x 475 columns


  reused fangraphs_pitching_2024.csv: 855 rows x 544 columns
  reused fangraphs_batting_2025.csv: 1470 rows x 475 columns


  reused fangraphs_pitching_2025.csv: 873 rows x 544 columns
  reused fangraphs_batting_2026.csv: 1472 rows x 475 columns


  reused fangraphs_pitching_2026.csv: 868 rows x 544 columns


## Player handedness (MLB Stats API)

`https://statsapi.mlb.com/api/v1/people?personIds=...` returns each player's `batSide` (L, R, or S for switch hitters) and `pitchHand`. The IDs are every MLBAM ID in the tables above, requested 100 at a time to keep URLs short.

This is the player's current listed handedness, not a record by season or plate appearance. Switch hitters are "S" even though they bat from one side in each plate appearance. Join on `player_id`, which matches Savant `player_id` and FanGraphs `xMLBAMID`.

IDs that MLB doesn't return are kept with `resolved = False` and empty handedness. On reruns, only IDs missing from the saved file are looked up and appended, and a check confirms existing rows are unchanged.

In [10]:
MLB_PEOPLE_URL = "https://statsapi.mlb.com/api/v1/people"
MLB_PEOPLE_BATCH_SIZE = 100
HANDEDNESS_COLUMNS = ["player_id", "full_name", "bat_side", "bat_side_description", "pitch_hand",
                      "pitch_hand_description", "primary_position", "resolved"]

savant_tables = [savant_custom_batter, savant_custom_pitcher, savant_xstats_batter,
                 savant_xstats_pitcher, savant_percentiles_batter, savant_percentiles_pitcher]
mlbam_ids = set()
for tables in savant_tables:
    for table in tables.values():
        mlbam_ids |= set(table["player_id"].astype(int))
for tables in [fangraphs_batting, fangraphs_pitching]:
    for table in tables.values():
        fg_ids = pd.to_numeric(table["xMLBAMID"], errors="coerce").dropna()
        mlbam_ids |= set(fg_ids[fg_ids > 0].astype(int))
mlbam_ids = sorted(mlbam_ids)
print(len(mlbam_ids), "distinct MLBAM player IDs across all season tables")


def download_handedness(player_ids):
    rows = []
    for start in range(0, len(player_ids), MLB_PEOPLE_BATCH_SIZE):
        batch = player_ids[start:start + MLB_PEOPLE_BATCH_SIZE]
        source = f"MLB people batch {start // MLB_PEOPLE_BATCH_SIZE + 1}"
        response = fetch(MLB_PEOPLE_URL, {"personIds": ",".join(map(str, batch))}, source)
        try:
            people = response.json()["people"]
        except (ValueError, KeyError) as error:
            raise ValueError(f"{source}: response was not the expected JSON ({error})") from error
        returned = [person["id"] for person in people]
        if set(returned) - set(batch) or len(returned) != len(set(returned)):
            raise ValueError(f"{source}: returned IDs that were not requested, or the same ID twice")
        for person in people:
            rows.append({"player_id": person["id"], "full_name": person.get("fullName"),
                         "bat_side": person.get("batSide", {}).get("code"),
                         "bat_side_description": person.get("batSide", {}).get("description"),
                         "pitch_hand": person.get("pitchHand", {}).get("code"),
                         "pitch_hand_description": person.get("pitchHand", {}).get("description"),
                         "primary_position": person.get("primaryPosition", {}).get("abbreviation"),
                         "resolved": True})
    resolved = {row["player_id"] for row in rows}
    rows += [{"player_id": player_id, "resolved": False} for player_id in player_ids if player_id not in resolved]
    return pd.DataFrame(rows, columns=HANDEDNESS_COLUMNS).sort_values("player_id").reset_index(drop=True)


def check_handedness(table, required_ids):
    problems = [f"missing columns {[c for c in HANDEDNESS_COLUMNS if c not in table.columns]}"]         if not set(HANDEDNESS_COLUMNS) <= set(table.columns) else []
    if not problems:
        if table.empty or table["player_id"].isna().any() or table["player_id"].duplicated().any():
            problems.append("no rows, or missing / duplicate player_id")
        uncovered = set(required_ids) - set(table["player_id"].astype(int))
        if uncovered:
            problems.append(f"{len(uncovered)} requested IDs absent, e.g. {sorted(uncovered)[:5]}")
    if problems:
        raise ValueError("handedness lookup: " + "; ".join(problems))


handedness_path = RAW_DIR / "mlb_player_handedness.csv"
if handedness_path.exists() and not OVERWRITE:
    saved = pd.read_csv(handedness_path)
    try:
        check_handedness(saved, set(saved["player_id"].dropna().astype(int)) if "player_id" in saved else set())
    except ValueError as error:
        raise ValueError(f"{handedness_path} exists but failed validation, so it was neither reused nor replaced: "
                         f"{error}. Inspect it, then delete it or set OVERWRITE = True.") from error
    new_ids = sorted(set(mlbam_ids) - set(saved["player_id"].astype(int)))
    if new_ids:
        added = download_handedness(new_ids)
        player_handedness = pd.concat([saved, added], ignore_index=True).sort_values("player_id").reset_index(drop=True)
        check_handedness(player_handedness, mlbam_ids)
        # The rows that were already saved must come through unchanged
        assert_same_values(player_handedness[player_handedness["player_id"].isin(saved["player_id"])], saved.sort_values("player_id"))
        save_csv(player_handedness, handedness_path)
        print(f"  extended {handedness_path.name}: {len(saved)} existing rows kept, {len(added)} new IDs added")
    else:
        player_handedness = saved
        print(f"  reused {handedness_path.name}: {len(saved)} rows")
else:
    player_handedness = download_handedness(mlbam_ids)
    check_handedness(player_handedness, mlbam_ids)
    save_csv(player_handedness, handedness_path)
    print(f"  saved  {handedness_path.name}: {player_handedness.shape[0]} rows x {player_handedness.shape[1]} columns")
player_handedness = pd.read_csv(handedness_path)

2635 distinct MLBAM player IDs across all season tables
  reused mlb_player_handedness.csv: 2635 rows


# Validation

These cells re-read the saved files and summarize coverage. Structural checks already ran when each file was saved or reused; these tables show the results together.

In [11]:
# Row and column counts by source, role and season
inventory = pd.DataFrame([{k: v for k, v in t.items() if k != "table"} for t in RAW_TABLES])
print("Rows")
display(inventory.pivot_table(index=["source", "role"], columns="season", values="rows", aggfunc="sum"))
print("Columns")
display(inventory.pivot_table(index=["source", "role"], columns="season", values="columns", aggfunc="sum"))
print(f"{len(inventory)} season tables + {handedness_path.name} ({len(player_handedness)} rows)")
leftover = sorted(p.name for p in RAW_DIR.glob("*.partial"))
print("unfinished .partial files:", leftover or "none")

Rows


season                           2022  2023  2024  2025  2026
source                  role                                 
FanGraphs               batter   1495  1457  1454  1470  1472
                        pitcher   871   863   855   873   868
Savant custom           batter    693   656   651   673   662
                        pitcher   813   805   806   812   819
Savant expected stats   batter    693   656   651   673   662
                        pitcher   871   863   855   873   868
Savant percentile ranks batter    628   614   606   617   612
                        pitcher   704   707   696   712   706

Columns


season                           2022  2023  2024  2025  2026
source                  role                                 
FanGraphs               batter    475   475   475   475   475
                        pitcher   544   544   544   544   544
Savant custom           batter    193   193   193   193   193
                        pitcher   317   317   317   317   317
Savant expected stats   batter     14    14    14    14    14
                        pitcher    17    17    17    17    17
Savant percentile ranks batter     23    23    23    23    23
                        pitcher    22    22    22    22    22

40 season tables + mlb_player_handedness.csv (2635 rows)
unfinished .partial files: none


In [12]:
# IDs, seasons and uniqueness, re-checked on the loaded tables
id_rows = []
for t in RAW_TABLES:
    table, id_column = t["table"], t["id_column"]
    season_column = "Season" if t["source"] == "FanGraphs" else "year"
    row = {"source": t["source"], "role": t["role"], "season": t["season"], "id_column": id_column,
           "id_dtype": str(table[id_column].dtype), "null_ids": int(table[id_column].isna().sum()),
           "duplicate_ids": int(table[id_column].duplicated().sum()),
           "seasons_in_file": ",".join(map(str, sorted(table[season_column].unique())))}
    if t["source"] == "FanGraphs":
        mlbam = pd.to_numeric(table["xMLBAMID"], errors="coerce")
        row["missing_xMLBAMID"] = int((mlbam.isna() | (mlbam <= 0)).sum())
        row["duplicate_xMLBAMID"] = int(mlbam[mlbam > 0].duplicated().sum())
    id_rows.append(row)
id_summary = pd.DataFrame(id_rows)
display(id_summary.groupby(["source", "role"]).agg(
    seasons=("seasons_in_file", ",".join), id_dtypes=("id_dtype", lambda s: ",".join(sorted(set(s)))),
    null_ids=("null_ids", "sum"), duplicate_ids=("duplicate_ids", "sum")))
fangraphs_ids = id_summary[id_summary["source"] == "FanGraphs"]
display(fangraphs_ids[["role", "season", "missing_xMLBAMID", "duplicate_xMLBAMID"]].set_index(["role", "season"]).T)
season_mismatch = id_summary[id_summary["seasons_in_file"] != id_summary["season"].astype(str)]
print("files whose season column differs from the requested season:", len(season_mismatch))

seasons id_dtypes  null_ids  \
source                  role                                                    
FanGraphs               batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   
Savant custom           batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   
Savant expected stats   batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   
Savant percentile ranks batter   2022,2023,2024,2025,2026     int64         0   
                        pitcher  2022,2023,2024,2025,2026     int64         0   

                                 duplicate_ids  
source                  role                    
FanGraphs               batter               0  
                        pitcher              0  
Savant custom           batter               0  
                        pitcher              0  
Savant expected stats   batter               0  
                        pitcher              0  
Savant percentile ranks batter               0  
                        pitcher              0

role,batter,pitcher,batter,pitcher,batter,pitcher,batter,pitcher,batter,pitcher
season,2022,2022,2023,2023,2024,2024,2025,2025,2026,2026
missing_xMLBAMID,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
duplicate_xMLBAMID,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


files whose season column differs from the requested season: 0


In [13]:
# Empty columns are reported, not rejected: some metrics don't exist in some seasons or on some boards.
# Savant field names were already checked against its schema, so these aren't typos.
missing_rows = []
for t in RAW_TABLES:
    table = t["table"]
    empty_columns = [c for c in table.columns if table[c].isna().all()]
    missing_rows.append({"source": t["source"], "role": t["role"], "season": t["season"],
                         "empty_cell_share": round(float(table.isna().mean().mean()), 3),
                         "entirely_empty_columns": len(empty_columns), "empty": empty_columns})
missing_summary = pd.DataFrame(missing_rows)
display(missing_summary.pivot_table(index=["source", "role"], columns="season",
                                    values="entirely_empty_columns", aggfunc="sum"))

# Empty columns by season, for the two broad sources
for source in ["Savant custom", "FanGraphs"]:
    for role in ["batter", "pitcher"]:
        subset = missing_summary[(missing_summary["source"] == source) & (missing_summary["role"] == role)]
        by_column = {}
        for _, row in subset.iterrows():
            for column in row["empty"]:
                by_column.setdefault(column, []).append(row["season"])
        patterns = {}
        for column, seasons in by_column.items():
            patterns.setdefault(tuple(seasons), []).append(column)
        print(f"\n{source} {role}: {len(by_column)} columns empty in at least one season")
        for seasons, columns in sorted(patterns.items(), key=lambda item: (-len(item[0]), item[0])):
            shown = ", ".join(columns[:12]) + (f", ... (+{len(columns) - 12} more)" if len(columns) > 12 else "")
            print(f"  empty in {list(seasons)}: {shown}")

season                           2022  2023  2024  2025  2026
source                  role                                 
FanGraphs               batter     80    26    26    34    25
                        pitcher    79    22    22    28    23
Savant custom           batter     12     0     0     0     0
                        pitcher    28     0     0     0     8
Savant expected stats   batter      0     0     0     0     0
                        pitcher     0     0     0     0     0
Savant percentile ranks batter      3     0     0     0     0
                        pitcher     1     1     1     1     1


Savant custom batter: 12 columns empty in at least one season
  empty in [2022]: swords, avg_swing_speed, avg_swing_length, squared_up_contact, squared_up_swing, blasts_contact, blasts_swing, fast_swing_rate, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path

Savant custom pitcher: 28 columns empty in at least one season
  empty in [2022, 2026]: n_sc_formatted, sc_avg_speed, sc_avg_spin, sc_avg_break_x, sc_avg_break_z, sc_avg_break_z_induced, sc_avg_break, sc_range_speed
  empty in [2022]: swords, avg_swing_speed, avg_swing_length, squared_up_contact, squared_up_swing, blasts_contact, blasts_swing, fast_swing_rate, attack_angle, attack_direction, ideal_angle_rate, vertical_swing_path, ... (+8 more)

FanGraphs batter: 82 columns empty in at least one season
  empty in [2022, 2023, 2024, 2025, 2026]: PO%, pfxFT%, pfxvFT, pfxFT-X, pfxFT-Z, pfxwFT, pfxwFT/C, pfxaaFT, pfxspFT
  empty in [2022, 2023, 2024, 2025]: BReview, BOverturned, BCorrect%, wBReview, PReview, POvert

In [14]:
# Savant custom field groups: share of rows with a value, averaged over each group's columns
for role, groups, tables in [("batter", BATTER_FIELD_GROUPS, savant_custom_batter),
                             ("pitcher", PITCHER_FIELD_GROUPS, savant_custom_pitcher)]:
    coverage = {}
    for season, table in tables.items():
        missing = [c for group in groups.values() for c in group if c not in table.columns]
        assert not missing, f"{role} {season}: missing requested columns {missing}"
        coverage[season] = {name: round(float(table[cols].notna().mean().mean()), 2) for name, cols in groups.items()}
    print(f"Savant custom {role}: all {sum(map(len, groups.values()))} requested columns present in every season")
    display(pd.DataFrame(coverage))

Savant custom batter: all 190 requested columns present in every season


,2022,2023,2024,2025,2026
profile,1.00,1.00,1.00,1.00,1.00
results,1.00,1.00,1.00,1.00,1.00
batting_counts,1.00,1.00,1.00,1.00,1.00
expected,0.99,0.99,0.99,0.99,0.99
contact_quality,0.99,0.99,0.99,0.99,0.99
batted_ball_profile,0.99,1.00,1.00,0.99,1.00
plate_discipline,0.95,0.99,1.00,1.00,1.00
bat_tracking,0.00,0.87,1.00,1.00,1.00
pitch_group_run_value,0.99,1.00,1.00,1.00,1.00
baserunning,0.94,0.94,0.94,0.94,0.94


Savant custom pitcher: all 314 requested columns present in every season


,2022,2023,2024,2025,2026
profile,1.00,1.00,1.00,1.00,1.00
workload_and_record,1.00,1.00,1.00,1.00,1.00
outcomes_allowed,1.00,1.00,1.00,1.00,1.00
pitch_result_counts,1.00,1.00,1.00,1.00,1.00
running_game,1.00,1.00,1.00,1.00,1.00
results_against,1.00,1.00,1.00,1.00,1.00
expected_against,1.00,1.00,1.00,1.00,1.00
contact_quality_against,1.00,1.00,1.00,1.00,1.00
batted_ball_profile_against,1.00,1.00,1.00,1.00,1.00
plate_discipline_induced,0.96,0.99,1.00,1.00,1.00


In [15]:
# FanGraphs field groups: filled share per season
FANGRAPHS_GROUPS = {
    "batter": {
        "identity_and_handedness": ["playerid", "xMLBAMID", "PlayerName", "TeamNameAbb", "Age", "Bats"],
        "standard": ["G", "PA", "AB", "H", "1B", "2B", "3B", "HR", "R", "RBI", "BB", "SO", "HBP", "SB", "AVG", "OBP", "SLG"],
        "advanced_value": ["wOBA", "wRC+", "WAR", "BABIP", "ISO", "Spd", "BaseRunning", "Offense"],
        "batted_ball": ["GB%", "LD%", "FB%", "IFFB%", "HR/FB", "Pull%", "Cent%", "Oppo%", "Soft%", "Hard%"],
        "plate_discipline": ["O-Swing%", "Z-Swing%", "Swing%", "O-Contact%", "Z-Contact%", "Contact%", "Zone%", "SwStr%", "CStr%"],
        "statcast_on_fangraphs": ["EV", "LA", "Barrel%", "HardHit%", "xwOBA", "xAVG", "xSLG"],
        "pitch_values_faced": ["wFB", "wSL", "wCB", "wCH", "wFB/C", "wSL/C"],
        "win_probability": ["WPA", "RE24", "REW", "pLI", "Clutch"],
    },
    "pitcher": {
        "identity_and_handedness": ["playerid", "xMLBAMID", "PlayerName", "TeamNameAbb", "Age", "Throws"],
        "standard": ["W", "L", "G", "GS", "SV", "HLD", "IP", "TBF", "H", "R", "ER", "HR", "BB", "SO", "ERA", "WHIP"],
        "advanced_value": ["FIP", "xFIP", "SIERA", "WAR", "K%", "BB%", "K-BB%", "LOB%", "BABIP"],
        "batted_ball": ["GB%", "LD%", "FB%", "HR/FB", "Soft%", "Hard%"],
        "plate_discipline": ["O-Swing%", "Z-Swing%", "O-Contact%", "Z-Contact%", "Zone%", "SwStr%", "C+SwStr%", "F-Strike%"],
        "statcast_on_fangraphs": ["EV", "LA", "Barrel%", "HardHit%", "xERA"],
        "pitch_mix_and_velocity": ["FB%", "FBv", "SL%", "SLv", "CH%", "CHv", "CB%", "CBv"],
        "stuff_plus": ["sp_stuff", "sp_location", "sp_pitching"],
        "win_probability": ["WPA", "RE24", "pLI", "gmLI"],
    },
}
for role, tables in [("batter", fangraphs_batting), ("pitcher", fangraphs_pitching)]:
    groups = FANGRAPHS_GROUPS[role]
    first = next(iter(tables.values()))
    absent = {name: [c for c in cols if c not in first.columns] for name, cols in groups.items()}
    absent = {name: cols for name, cols in absent.items() if cols}
    print(f"FanGraphs {role}: checked key columns absent from the source:", absent or "none")
    coverage = {season: {name: round(float(table[[c for c in cols if c in table.columns]].notna().mean().mean()), 2)
                         for name, cols in groups.items()} for season, table in tables.items()}
    display(pd.DataFrame(coverage))

FanGraphs batter: checked key columns absent from the source: none


,2022,2023,2024,2025,2026
identity_and_handedness,1.00,1.00,1.00,1.00,1.00
standard,1.00,1.00,1.00,1.00,1.00
advanced_value,0.93,0.93,0.93,0.93,0.93
batted_ball,0.73,0.72,0.72,0.73,0.72
plate_discipline,0.46,0.45,0.45,0.46,0.45
statcast_on_fangraphs,0.46,0.45,0.45,0.45,0.45
pitch_values_faced,0.46,0.44,0.44,0.45,0.44
win_probability,0.46,0.45,0.45,0.46,0.45


FanGraphs pitcher: checked key columns absent from the source: none


,2022,2023,2024,2025,2026
identity_and_handedness,1.00,1.00,1.00,1.00,1.00
standard,1.00,1.00,1.00,1.00,1.00
advanced_value,1.00,1.00,1.00,1.00,1.00
batted_ball,1.00,1.00,1.00,1.00,1.00
plate_discipline,1.00,1.00,1.00,1.00,1.00
statcast_on_fangraphs,1.00,1.00,1.00,1.00,1.00
pitch_mix_and_velocity,0.75,0.73,0.73,0.73,0.75
stuff_plus,1.00,1.00,1.00,1.00,1.00
win_probability,1.00,1.00,1.00,1.00,1.00


In [16]:
# Savant custom vs expected stats: who is on only one list. Their inclusion rules differ.
coverage_rows, xstats_only_examples = [], {}
for role, custom, xstats in [("batter", savant_custom_batter, savant_xstats_batter),
                             ("pitcher", savant_custom_pitcher, savant_xstats_pitcher)]:
    for season in SEASONS:
        custom_ids, xstats_ids = set(custom[season]["player_id"]), set(xstats[season]["player_id"])
        only_x = xstats[season][xstats[season]["player_id"].isin(xstats_ids - custom_ids)]
        positions = player_handedness.set_index("player_id").reindex(only_x["player_id"])["primary_position"]
        coverage_rows.append({"role": role, "season": season, "custom": len(custom_ids), "expected_stats": len(xstats_ids),
                              "both": len(custom_ids & xstats_ids), "custom_only": len(custom_ids - xstats_ids),
                              "expected_only": len(only_x),
                              "expected_only_pa_range": f"{only_x['pa'].min()}-{only_x['pa'].max()}" if len(only_x) else "",
                              "expected_only_non_pitchers": int((positions != "P").sum()) if role == "pitcher" else None})
        xstats_only_examples[(role, season)] = only_x
display(pd.DataFrame(coverage_rows).set_index(["role", "season"]))
print("Pitchers in expected stats but not on the custom board are mostly position players who pitched "
      "(primary_position != 'P' in the MLB lookup); the rest are true pitchers with very few batters faced. "
      "Expected stats therefore add these players' expected-stat rows only. They are not added to the custom table.")

custom  expected_stats  both  custom_only  expected_only  \
role    season                                                             
batter  2022       693             693   693            0              0   
        2023       656             656   656            0              0   
        2024       651             651   651            0              0   
        2025       673             673   673            0              0   
        2026       662             662   662            0              0   
pitcher 2022       813             871   813            0             58   
        2023       805             863   805            0             58   
        2024       806             855   806            0             49   
        2025       812             873   812            0             61   
        2026       819             868   819            0             49   

               expected_only_pa_range  expected_only_non_pitchers  
role    season                                                     
batter  2022                                                  NaN  
        2023                                                  NaN  
        2024                                                  NaN  
        2025                                                  NaN  
        2026                                                  NaN  
pitcher 2022                     1-47                        56.0  
        2023                     1-32                        55.0  
        2024                     1-20                        46.0  
        2025                     1-54                        61.0  
        2026                     2-49                        48.0

Pitchers in expected stats but not on the custom board are mostly position players who pitched (primary_position != 'P' in the MLB lookup); the rest are true pitchers with very few batters faced. Expected stats therefore add these players' expected-stat rows only. They are not added to the custom table.


In [17]:
# FanGraphs vs Savant: ID overlap, and whether matched rows agree. Both are regular season with
# teams combined. PA / TBF are compared with Savant's standard counts (b_total_pa / p_total_pa) and pa.
def ip_to_outs(innings):
    """Convert baseball innings notation to outs: 61.1 is 184 outs. Only .0, .1 and .2 are valid."""
    innings = pd.to_numeric(innings)
    whole = innings // 1
    extra = ((innings - whole) * 10).round()
    if not extra.dropna().isin([0, 1, 2]).all():
        raise ValueError("found an innings value with .3 or higher, so this is not baseball innings notation")
    return whole * 3 + extra


overlap_rows = []
for role, fangraphs, savant in [("batter", fangraphs_batting, savant_custom_batter),
                                ("pitcher", fangraphs_pitching, savant_custom_pitcher)]:
    for season in SEASONS:
        fg = fangraphs[season].copy()
        fg["mlbam"] = pd.to_numeric(fg["xMLBAMID"], errors="coerce")
        if role == "batter":
            fg = fg[fg["PA"] > 0]       # the FanGraphs batting board also lists pitchers with 0 PA
        sv = savant[season]
        matched = fg.merge(sv, left_on="mlbam", right_on="player_id", how="inner", validate="one_to_one")
        row = {"role": role, "season": season, "fangraphs_rows": len(fg), "savant_rows": len(sv), "matched": len(matched),
               "fangraphs_only": int((~fg["mlbam"].isin(sv["player_id"])).sum()),
               "savant_only": int((~sv["player_id"].isin(fg["mlbam"])).sum())}
        if role == "batter":
            row["PA_eq_b_total_pa"] = round(float((matched["PA"] == matched["b_total_pa"]).mean()), 3)
            row["PA_eq_pa"] = round(float((matched["PA"] == matched["pa"]).mean()), 3)
            row["max_abs_PA_diff"] = int((matched["PA"] - matched["b_total_pa"]).abs().max())
        else:
            row["TBF_eq_p_total_pa"] = round(float((matched["TBF"] == matched["p_total_pa"]).mean()), 3)
            row["TBF_eq_pa"] = round(float((matched["TBF"] == matched["pa"]).mean()), 3)
            row["max_abs_TBF_diff"] = int((matched["TBF"] - matched["p_total_pa"]).abs().max())
            row["outs_equal_share"] = round(float((ip_to_outs(matched["IP"]) == ip_to_outs(matched["p_formatted_ip"])).mean()), 3)
        fg_only = fg[~fg["mlbam"].isin(sv["player_id"])]
        size_column = "PA" if role == "batter" else "TBF"
        row["fangraphs_only_max_" + size_column] = fg_only[size_column].max() if len(fg_only) else None
        overlap_rows.append(row)
display(pd.DataFrame(overlap_rows).set_index(["role", "season"]))

fangraphs_rows  savant_rows  matched  fangraphs_only  \
role    season                                                         
batter  2022               693          693      693               0   
        2023               656          656      656               0   
        2024               651          651      651               0   
        2025               673          673      673               0   
        2026               662          662      662               0   
pitcher 2022               871          813      813              58   
        2023               863          805      805              58   
        2024               855          806      806              49   
        2025               873          812      812              61   
        2026               868          819      819              49   

                savant_only  PA_eq_b_total_pa  PA_eq_pa  max_abs_PA_diff  \
role    season                                                             
batter  2022              0             1.000     0.993              0.0   
        2023              0             1.000     0.988              0.0   
        2024              0             1.000     1.000              0.0   
        2025              0             0.999     0.990              1.0   
        2026              0             0.997     0.985              1.0   
pitcher 2022              0               NaN       NaN              NaN   
        2023              0               NaN       NaN              NaN   
        2024              0               NaN       NaN              NaN   
        2025              0               NaN       NaN              NaN   
        2026              0               NaN       NaN              NaN   

                fangraphs_only_max_PA  TBF_eq_p_total_pa  TBF_eq_pa  \
role    season                                                        
batter  2022                      NaN                NaN        NaN   
        2023                      NaN                NaN        NaN   
        2024                      NaN                NaN        NaN   
        2025                      NaN                NaN        NaN   
        2026                      NaN                NaN        NaN   
pitcher 2022                      NaN                1.0      0.977   
        2023                      NaN                1.0      0.983   
        2024                      NaN                1.0      0.980   
        2025                      NaN                1.0      0.964   
        2026                      NaN                1.0      0.977   

                max_abs_TBF_diff  outs_equal_share  fangraphs_only_max_TBF  
role    season                                                              
batter  2022                 NaN               NaN                     NaN  
        2023                 NaN               NaN                     NaN  
        2024                 NaN               NaN                     NaN  
        2025                 NaN               NaN                     NaN  
        2026                 NaN               NaN                     NaN  
pitcher 2022                 0.0               1.0                    47.0  
        2023                 0.0               1.0                    32.0  
        2024                 0.0               1.0                    20.0  
        2025                 0.0               1.0                    54.0  
        2026                 0.0               1.0                    49.0

In [18]:
# Handedness lookup coverage, and agreement with the season tables
resolved = player_handedness["resolved"].astype(bool)
print(f"{len(player_handedness)} IDs looked up, {int(resolved.sum())} resolved, {int((~resolved).sum())} unresolved")
if (~resolved).any():
    print("  unresolved IDs:", player_handedness.loc[~resolved, "player_id"].tolist()[:20])
print("bat_side:", player_handedness["bat_side"].value_counts(dropna=False).to_dict(),
      "| pitch_hand:", player_handedness["pitch_hand"].value_counts(dropna=False).to_dict())

lookup = player_handedness.set_index("player_id")
pitch_hand = pd.concat([t[["player_id", "pitch_hand"]] for t in savant_custom_pitcher.values()]).drop_duplicates()
pitch_hand = pitch_hand.join(lookup["pitch_hand"].rename("mlb_pitch_hand"), on="player_id")
print(f"Savant pitch_hand vs MLB pitchHand: {int((pitch_hand['pitch_hand'] != pitch_hand['mlb_pitch_hand']).sum())} "
      f"disagreements in {len(pitch_hand)} pitcher rows")

# FanGraphs writes switch hitters as "B" (both); MLB uses "S"
fg_bats = pd.concat([t[["xMLBAMID", "Bats"]] for t in fangraphs_batting.values()]).drop_duplicates()
fg_bats = fg_bats.assign(Bats=fg_bats["Bats"].replace({"B": "S"})).join(lookup["bat_side"], on="xMLBAMID")
disagree = fg_bats[fg_bats["Bats"] != fg_bats["bat_side"]].dropna()
print(f"FanGraphs Bats vs MLB batSide: {len(disagree)} disagreements in {len(fg_bats)} batter rows",
      disagree.head(5).to_dict("records") if len(disagree) else "")

2635 IDs looked up, 2635 resolved, 0 unresolved
bat_side: {'R': 1695, 'L': 803, 'S': 137} | pitch_hand: {'R': 2110, 'L': 523, 'S': 2}
Savant pitch_hand vs MLB pitchHand: 0 disagreements in 1520 pitcher rows


FanGraphs Bats vs MLB batSide: 1 disagreements in 2635 batter rows [{'xMLBAMID': 666801, 'Bats': 'S', 'bat_side': 'R'}]


In [19]:
# CSV round trip: files on disk match this run's tables, and IDs read back as integers
problems = []
for t in RAW_TABLES + [{"file": handedness_path.name, "table": player_handedness, "id_column": "player_id"}]:
    on_disk = pd.read_csv(RAW_DIR / t["file"])
    try:
        assert_same_values(on_disk, t["table"])
    except AssertionError as error:
        problems.append(f"{t['file']}: {str(error)[:120]}")
    if not pd.api.types.is_integer_dtype(on_disk[t["id_column"]]):
        problems.append(f"{t['file']}: {t['id_column']} read back as {on_disk[t['id_column']].dtype}")
print(f"round-trip checked {len(RAW_TABLES) + 1} files:", problems or "all identical, IDs read back as integers")

round-trip checked 41 files: all identical, IDs read back as integers


In [20]:
# 0 when every file is already saved and OVERWRITE = False
print(f"network requests this run: {len(requests_made)}")
if requests_made:
    print(pd.Series([s.rsplit(" ", 1)[0] if s.startswith("MLB") else " ".join(s.split()[:3]) for s in requests_made]).value_counts().to_dict())

network requests this run: 0
